In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.8 MB/s eta 0:00:00


In [3]:
import os
import yaml
import cv2
from ultralytics import YOLO

def setup_kaggle_yaml(base_path):
    """
    Rewrites the dataset's data.yaml into the writable /kaggle/working/ directory.
    This prevents Kaggle's Read-Only file system from crashing the training process.
    """
    original_yaml = os.path.join(base_path, "data.yaml")
    working_yaml = "/kaggle/working/custom_data.yaml"
    
    with open(original_yaml, 'r') as f:
        data = yaml.safe_load(f)
        
    # Force absolute paths pointing to the Kaggle mount
    data['path'] = base_path
    data['train'] = "train/images"
    data['val'] = "valid/images"
    data['test'] = "test/images"
    
    with open(working_yaml, 'w') as f:
        yaml.dump(data, f)
        
    return working_yaml


def train_model(data_yaml_path):
    model = YOLO('yolov8s.pt') 
    
    model.train(
        data=data_yaml_path,
        epochs=30,             
        imgsz=1024, 
        batch=16,              
        #This array syntax commands YOLO to use both T4 GPUs via DataParallel
        device=[0, 1],         
        workers=4,             
        project='/kaggle/working/Drone_Rescue_Project', 
        name='SARD_YOLO_Run'
    )


# VIDEO INFERENCE

def run_inference(model_path, input_video, output_video):
    model = YOLO(model_path)
    cap = cv2.VideoCapture(input_video)
    
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        results = model(frame, imgsz=1024, conf=0.35)[0]
        out.write(results.plot())
            
    cap.release()
    out.release()
    cv2.destroyAllWindows()

if __name__ == '__main__':
    
    DATASET_BASE_PATH = "/kaggle/input/datasets/nikolasgegenava/sard-search-and-rescue/search-and-rescue" 
    
    print(f"[INFO] Setting up dataset config from: {DATASET_BASE_PATH}")
    working_yaml = setup_kaggle_yaml(DATASET_BASE_PATH.strip())
    
    print("[INFO] Starting Multi-GPU YOLO Training...")
    train_model(working_yaml)
    
    print("[INFO] Running Inference...")
    best_weights = "/kaggle/working/Drone_Rescue_Project/SARD_YOLO_Run/weights/best.pt"
    
    

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[INFO] Setting up dataset config from: /kaggle/input/datasets/nikolasgegenava/sard-search-and-rescue/search-and-rescue
[INFO] Starting Multi-GPU YOLO Training...
Ultralytics 8.4.159 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/custom_data.yaml, degrees=0.0

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      5.31G      1.957      1.843      1.779         11       1024: 100% ━━━━━━━━━━━━ 253/253 3.6it/s 1:100.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 3.1it/s 11.6s0.3s
                   all       1144       1463     0.0797      0.589     0.0629     0.0248

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      5.32G      1.983      1.831      1.784          9       1024: 100% ━━━━━━━━━━━━ 253/253 3.6it/s 1:100.3s3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 3.2it/s 11.2s0.3s
                   all       1144       1463      0.691      0.497      0.558      0.237

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      5.32G      1.898      1.749      1.732          5       